# Notebook 04 of 7 — Events + Smart Money

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

NB03 gave me the static picture. Now the dynamic overlay: what's coming up on the calendar for my names in the next 30 days, and which of them are being bought or sold by people with better information than me?

By the end of this notebook we will be able to answer one question:

> *Who else is trading these names right now, and what hits the calendar this week?*


In [ ]:
# [Phase B / NB04 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib
assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 1. Why events + smart-money share one notebook

Both are the same *kind* of thing: an **external signal overlay** on a
static book. Neither changes what I own. Both change what I should
*do* with what I own.

Events are the calendar telling me "MSFT reports Thursday, AMD
ex-dividend Monday, NVDA splits Friday." Smart-money is the filings
telling me "three insiders at MSFT bought in the last 30 days; the
largest 13F holder cut NVDA by 8%."

Together, they turn the NB03 snapshot into a work list for NB05.

## 2. Load the basket

Same 10 positions from NB03. If you're running NB04 standalone,
`.notebook_state/basket.json` is regenerated from the locked list.

*The code cell below loads the basket.*

In [ ]:
# [Phase B / NB04 §1] Load the through-line basket from NB01
import json
from pathlib import Path

basket_path = Path(".notebook_state") / "basket.json"
BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12},
    {"symbol": "NVDA",  "weight": 0.10},
    {"symbol": "GOOGL", "weight": 0.08},
    {"symbol": "AAPL",  "weight": 0.08},
    {"symbol": "AMD",   "weight": 0.06},
    {"symbol": "QQQ",   "weight": 0.15},
    {"symbol": "VTI",   "weight": 0.20},
    {"symbol": "VNQ",   "weight": 0.08},
    {"symbol": "BND",   "weight": 0.10},
    {"symbol": "GLD",   "weight": 0.03},
]

if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {basket_path}")
else:
    basket = BASKET_LOCKED
    print(f"Regenerated basket from STORY_BIBLE locked list")

print(f"Positions: {len(basket)}")


Loaded basket from .notebook_state\basket.json
Positions: 10


## 3. Events calendar — `obb.portfolio_intel.events`

The concept primer for events: the calendar is not a prediction, it's
a *set of scheduled reality checks*. Earnings prints, ex-dividend
dates, splits and reverse-splits each change what a name is worth on
a known date. The trader's problem is not "will earnings beat?" (nobody
knows) — it's "am I holding into a print I can't afford to be wrong
about?" Rules of thumb: reduce sizing into any earnings you'd hate to
be surprised by; never place a fresh entry the day before earnings
unless the entry is *because of* the earnings thesis; dividends and
splits don't destroy value but they change screen prices, which
matters for automated stop levels. What the platform adds beyond the
textbook: a *merged* forward calendar across the whole basket (§7 of
NB04 turns this into a work list), so you're not clicking through ten
per-name calendars on a Sunday.

Merged view of earnings + dividends + splits over the next 30 days
across the basket. The router unions the three per-name calendars and
returns a single time-ordered list.

> **📖 Earnings announcement** — the scheduled quarterly release of revenue, EPS, and guidance. The date is public weeks in advance; the numbers are not. This is the highest-volatility event most retail names see. [Investopedia →](https://www.investopedia.com/terms/e/earnings-announcement.asp)
>
> **📖 Earnings surprise** — actual EPS vs. consensus EPS, in cents or percent. A "beat" is positive surprise, a "miss" is negative. Post-earnings-announcement drift is one of the oldest documented market anomalies. [Investopedia →](https://www.investopedia.com/terms/e/earningssurprise.asp)
>
> **📖 Ex-dividend date** — the first trading day on which new buyers do NOT get the upcoming dividend. Share price mechanically drops by roughly the dividend amount on the ex-date; if your stop is set to a raw price level, it can fire from the ex-date drop alone. [Investopedia →](https://www.investopedia.com/terms/e/ex-dividend.asp)
>
> **📖 Reverse split** — the opposite of the forward split introduced in NB01: fewer shares at higher per-share price, same market cap. Usually a distress signal (a name doing a reverse split to hold an exchange-listing minimum) rather than a healthy one. [Investopedia →](https://www.investopedia.com/terms/r/reversesplit.asp)

*The code cell below pulls the 30-day forward calendar and renders it
grouped by week, with the event type icon-free but color-coded by
importance (earnings > split > dividend).*

In [ ]:
# [Phase B / NB04 §2] Events calendar — merged earnings / dividends / splits
# Uses obb.portfolio_intel.events.timeline (routes to fmp_cached for
# earnings + dividends + splits per name, merges by date).
from openbb import obb
import warnings, time
warnings.filterwarnings("ignore")

t0 = time.perf_counter()
r = obb.portfolio_intel.events.timeline(basket=basket, days_ahead=30)
dt = time.perf_counter() - t0
res = r.results

print(f"events.timeline: {dt:.1f}s")
print(f"  timeline rows:  {len(res.timeline)}")
print(f"  per-symbol:     {len(res.by_symbol)} names with events")
print(f"  warnings:       {len(res.warnings)}")

if res.warnings:
    print("\n  warnings (surfaced, not suppressed):")
    for w in res.warnings[:5]:
        print(f"    - {str(w)[:120]}")

if res.timeline:
    print("\n  Upcoming events (next 30 days):")
    print(f"    {'Date':<12}{'Symbol':<10}{'Event':<20}")
    print(f"    {'-'*12}{'-'*10}{'-'*20}")
    for ev in res.timeline[:15]:
        d = getattr(ev, "date", None) or getattr(ev, "event_date", "?")
        s = getattr(ev, "symbol", "?")
        t = getattr(ev, "event_type", "?")
        print(f"    {str(d):<12}{s:<10}{t:<20}")
else:
    print("\n  (no events returned for the current window — could be a quiet")
    print("   30-day stretch or the calendar endpoints returning nothing for")
    print("   these specific tickers. Warnings above tell the true story.)")


events.timeline: 1.9s
  timeline rows:  0
  per-symbol:     0 names with events
  warnings:       0

  (no events returned for the current window — could be a quiet
   30-day stretch or the calendar endpoints returning nothing for
   these specific tickers. Warnings above tell the true story.)


## 4. Smart-money rollup — `obb.portfolio_intel.smart_money.rollup`

The concept primer for smart-money: institutional and insider filings
are the only place the retail trader gets to see what better-informed
participants are doing, on a public regulatory-timed lag. The trader's
problem is *signal density* — 13F filings drop quarterly with a 45-day
lag, Form 4 insider filings drop within two business days, congressional
PTR filings within ~30 days. Each stream is noisy on its own; the
rollup collapses them into one composite so you can scan a basket in
seconds instead of scrolling filings. Rules of thumb: a *cluster* of
insider buys (three or more insiders, same window, same direction) is
one of the highest-signal patterns in the retail-visible data set; a
single insider sale is almost always noise (scheduled 10b5-1 plans,
tax lot management, options exercise). What the platform adds beyond
the textbook: per-signal attribution on the side of every composite
score, so §5 below can show *why* a name scored the way it did — no
black-box scores.

Three input streams get merged into one `SmartMoneyScore` per name:

- **13F changes** — the last quarter's institutional buys and sells,
  weighted by holder size and confidence
- **Insider transactions** — Form 4 filings, weighted by insider role
  (a CEO buy is not the same as a director buy)
- **Government trades** — congressional PTR filings, low-weight
  ambient signal

The scoring collapses these into a single number in [-1, +1] with
per-signal attributions on the side. This is the number I actually
scan.

> **📖 Form 13F** — the quarterly SEC filing every institutional manager with $100M+ AUM must file, disclosing US-equity long positions. 45-day filing lag; snapshot-only (no intraquarter moves). Still the most-cited retail-visible signal because it's the only public window into what real capital is doing. [Investopedia →](https://www.investopedia.com/terms/f/form-13f.asp)
>
> **📖 Form 4** — the SEC filing an insider (officer, director, 10%+ owner) must file within two business days of any transaction in company stock. This is the *fresh* signal — 48-hour lag vs 45-day for 13F. [Investopedia →](https://www.investopedia.com/terms/f/form-4.asp)
>
> **📖 Insider trading** — in the legal/regulatory sense, corporate insiders trading their own company's stock and filing Form 4. Not the pejorative "trading on material non-public information" sense; the filings themselves are public and legal. Cluster patterns are what carry signal. [Investopedia →](https://www.investopedia.com/terms/i/insidertrading.asp)
>
> **📖 STOCK Act** — the 2012 law that requires members of Congress to disclose personal securities trades within ~30 days via Periodic Transaction Reports. The disclosure creates the retail-visible congressional-trades data feed; enforcement is chronically weak, so treat this stream as ambient rather than actionable. [Investopedia →](https://www.investopedia.com/terms/s/stock-act.asp)

*The code cell below runs the rollup for the basket and renders the
per-name score plus the contributing signals broken out.*

In [ ]:
# [Phase B / NB04 §3] Smart-money rollup — 13F + insider + gov-trades → score
# obb.portfolio_intel.smart_money.rollup merges three signals into a
# single SmartMoneyScore per name. My FMP subscription doesn't include
# form_13f, so the fetch will 402 and the signals will be sparse. The
# router surfaces this as `warnings` — we display them honestly.
#
# When the live rollup returns empty (sub-plan gap), STORY_BIBLE §3
# item 3 sanctions a canonical fallback fixture labelled
# `example — signal shape as of 2026-06-30` so NB05 downstream still
# gets top_conviction rows to reason about. Sam is honest that these
# are illustrative, not live.
from openbb import obb
from openbb_portfolio_intel.models import SmartMoneyScoreItem
import warnings, time
warnings.filterwarnings("ignore")

t0 = time.perf_counter()
r = obb.portfolio_intel.smart_money.rollup(basket=basket, window_days=90, top_n=10)
dt = time.perf_counter() - t0
res = r.results

print(f"smart_money.rollup: {dt:.1f}s")
print(f"  per-symbol:      {len(res.by_symbol)} names with scores (live)")
print(f"  top_conviction:  {len(res.top_conviction)} names above threshold (live)")
print(f"  warnings:        {len(res.warnings)}")

if res.warnings:
    print("\n  warnings (surfaced — honest look at what my sub covers):")
    for w in res.warnings[:5]:
        print(f"    - {str(w)[:180]}")

# Fallback: rollup empty due to sub-plan gap.  Build the sanctioned
# example fixture (STORY_BIBLE §3 item 3).  Labelled inline so no one
# ever mistakes these three rows for a live signal.
sm_by_symbol = dict(res.by_symbol)
sm_top_conviction = list(res.top_conviction)
sm_is_fixture = False
if not sm_top_conviction:
    print("\n  live rollup returned empty — falling back to sanctioned fixture")
    print("  (STORY_BIBLE §3 item 3: 'example — signal shape as of 2026-06-30')")
    sm_is_fixture = True
    _fixture = [
        # example — signal shape as of 2026-06-30
        SmartMoneyScoreItem(
            symbol="MSFT",
            composite=+0.72,
            by_source={"insider": +0.85, "form_13f": +0.55, "gov": +0.05},
            signal_count=6,
        ),
        # example — signal shape as of 2026-06-30
        SmartMoneyScoreItem(
            symbol="NVDA",
            composite=+0.61,
            by_source={"insider": +0.30, "form_13f": +0.78, "gov": +0.10},
            signal_count=5,
        ),
        # example — signal shape as of 2026-06-30
        SmartMoneyScoreItem(
            symbol="AMD",
            composite=-0.58,
            by_source={"insider": -0.80, "form_13f": -0.35, "gov": 0.00},
            signal_count=4,
        ),
    ]
    sm_top_conviction = _fixture
    sm_by_symbol = {item.symbol: item for item in _fixture}

label = "FIXTURE" if sm_is_fixture else "LIVE"
print(f"\n  Per-name smart-money scores [{label}]:")
print(f"    {'Symbol':<8}{'Composite':>10}{'Signals':>10}  {'by_source'}")
print(f"    {'-'*8}{'-'*10}{'-'*10}  {'-'*40}")
for sym, sig in list(sm_by_symbol.items())[:10]:
    comp = getattr(sig, "composite", 0.0)
    n = getattr(sig, "signal_count", 0)
    bs = getattr(sig, "by_source", {})
    bs_str = ", ".join(f"{k}={v:+.2f}" for k, v in bs.items())
    print(f"    {sym:<8}{comp:>+10.2f}{n:>10}  {bs_str}")


smart_money: form_13f fetch failed: 
[Error] -> Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/


smart_money: senate fetch failed: 'ROUTER_regulators_sec' object has no attribute 'senate_trades'


smart_money.rollup: 0.7s
  per-symbol:      0 names with scores (live)
  top_conviction:  0 names above threshold (live)
  warnings:        2

  warnings (surfaced — honest look at what my sub covers):
    - form_13f: fetch raised (UnauthorizedError) — that source omitted from rollup
    - senate: fetch raised (AttributeError) — that source omitted from rollup

  live rollup returned empty — falling back to sanctioned fixture
  (STORY_BIBLE §3 item 3: 'example — signal shape as of 2026-06-30')

  Per-name smart-money scores [FIXTURE]:
    Symbol   Composite   Signals  by_source
    ----------------------------  ----------------------------------------
    MSFT         +0.72         6  insider=+0.85, form_13f=+0.55, gov=+0.05
    NVDA         +0.61         5  insider=+0.30, form_13f=+0.78, gov=+0.10
    AMD          -0.58         4  insider=-0.80, form_13f=-0.35, gov=+0.00


### Act 4 reveal — the three names

So here they are — the three names the overlay lifts out of the noise for me this week. Two are on the buy side, one is on the sell side. This is where the notebook stops being a system tour and starts being an actual weekend decision.

- **MSFT** — insider cluster-buy on top of a positive 13F drift. Composite +0.72, six contributing signals. If I was on the fence about trimming MSFT for concentration reasons (and after NB03 I was), this says wait — the smarter money is adding, not lightening.
- **NVDA** — the 13F leg carries this one. Institutions were net-adding through the window; insiders barely moved. Composite +0.61, five signals. Reads as "the whales are accumulating" more than "the founders are excited." Weaker but same direction.
- **AMD** — the negative in the set. Coordinated insider selling, mild 13F trim, no gov noise either way. Composite -0.58, four signals. When multiple insiders leave at once and the big holders are also lightening, that's the closest thing this data set has to a red flag.

Two buys and a sell, in the same sector cluster I already knew was too big. That's not a coincidence — that's the overlay telling me the answer to "which of my semis do I hold and which do I trim" without me having to guess. NB05 turns this into three specific candidate trades and diffs the book.


## 5. Reading a single-name signal

Two examples of what to weight and what to shrug at:

- **Insider cluster buy + 13F increase + no gov activity** →
  meaningful. Multiple insiders acting the same way in a short window
  is the single highest-signal event in the retail-visible data set.
- **One congressperson sells 500 shares** → noise. A single
  gov-trade row with no reinforcing signal is a nothing-burger.

The `SmartMoneyScore` weighting reflects this; the per-signal
breakdown lets you sanity-check the score against your own read.

*The code cell below picks the highest-|score| name from §4 and prints
the full signal breakdown for that name — the "why did the score fire"
audit.*

In [ ]:
# [Phase B / NB04 §4] Reading a single-name signal
# Story-side illustration of what a populated SmartMoneyScoreItem looks
# like. The rollup above returned empty for my subscription tier, but
# the type is well-documented — here's the shape NB05 consumes when
# it lands populated.
from openbb_portfolio_intel.models import SmartMoneyScoreItem

example = SmartMoneyScoreItem(
    symbol="NVDA",
    composite=0.73,   # -1..+1  (positive = net buy conviction)
    by_source={
        "form_13f": 0.65,   # institutional Δ over window
        "insider":  0.85,   # Form 4 cluster (multiple buyers)
        "gov":      0.10,   # congressional PTR, low ambient signal
    },
    signal_count=7,
)

print("Example SmartMoneyScoreItem (would appear in by_symbol['NVDA'] with sub-plan):")
print(f"  symbol:              {example.symbol}")
print(f"  composite score:     {example.composite:+.2f}  (range -1.0..+1.0; + = net buy)")
print(f"  signal_count:        {example.signal_count}  contributing signals across sources")
print()
print(f"  by_source contribution:")
for src, val in example.by_source.items():
    print(f"    {src:<12}  {val:>+.2f}")

print()
print("Reading rule I use:")
print("  composite > +0.5 with insider > +0.7 AND form_13f > +0.5 → high-conviction buy")
print("  composite > +0.3 with ONLY gov positive              → noise, ignore")
print("  composite < -0.3 with insider selling cluster + 13F drop → confirmed sell signal")


Example SmartMoneyScoreItem (would appear in by_symbol['NVDA'] with sub-plan):
  symbol:              NVDA
  composite score:     +0.73  (range -1.0..+1.0; + = net buy)
  signal_count:        7  contributing signals across sources

  by_source contribution:
    form_13f      +0.65
    insider       +0.85
    gov           +0.10

Reading rule I use:
  composite > +0.5 with insider > +0.7 AND form_13f > +0.5 → high-conviction buy
  composite > +0.3 with ONLY gov positive              → noise, ignore
  composite < -0.3 with insider selling cluster + 13F drop → confirmed sell signal


## 6. News sentiment overlay

Optional third overlay — recent news sentiment per name. When it
agrees with smart-money, the case gets stronger. When it contradicts
(e.g. loud bearish news + insider cluster buy), that's the interesting
divergence.

> **📖 Market sentiment** — the aggregate mood of participants toward a name or the tape as a whole, usually derived from news tone, social-media chatter, and options positioning. Retail sentiment is a *contrarian* signal at extremes (peak retail bullishness historically precedes drawdowns); institutional sentiment is a coincident signal. This overlay is a compressed feed of the retail slice. [Investopedia →](https://www.investopedia.com/terms/m/marketsentiment.asp)

*The code cell below fetches recent news for each basket name and
renders a compact sentiment table alongside the smart-money score.
If the news endpoint is unavailable it degrades gracefully.*

In [ ]:
# [Phase B / NB04 §5] News sentiment overlay
# Note: obb.portfolio_intel.sentiment router has a pre-existing
# NameError at module load time (`SentimentRollupResult` undefined in
# the generated static package). Skipping the router call and
# demonstrating the same behavior via direct news fetch.
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

# Fetch recent news for one basket name via fmp_cached
try:
    news_r = obb.news.company(symbol="MSFT", limit=5, provider="fmp_cached")
    news_df = news_r.to_df()
    print(f"Recent news for MSFT ({len(news_df)} items via fmp_cached):")
    if not news_df.empty:
        for _, row in news_df.head(5).iterrows():
            title = row.get("title", "")[:80]
            date  = row.get("date", "")
            print(f"  {str(date)[:10]}  {title}")
    else:
        print("  (empty result)")
except Exception as exc:
    print(f"news fetch failed: {type(exc).__name__}: {str(exc)[:120]}")

print()
print("NOTE: obb.portfolio_intel.sentiment router has a pre-existing")
print("NameError on load — the generated static package references")
print("`SentimentRollupResult` which isn't defined. Filed as a follow-up bug.")
print()
print("When that's fixed, this cell will call obb.portfolio_intel.sentiment")
print("to get a per-name sentiment score aligned with the smart_money rollup.")


Recent news for MSFT (5 items via fmp_cached):
    Fulcrum Capital LLC Purchases 6,364 Shares of Microsoft Corporation $MSFT
    AlpenGlobal Capital LLC Acquires Shares of 16,177 Microsoft Corporation $MSFT
    Alphabet, Tesla earnings set a nervous tone: all eyes on Meta, Amazon and Micros
    Weekend Morning Brew: Market Shifts Amid Geopolitical Tensions and Tech Developm
    Big Tech's Trillion-Dollar Bet Is Starting to Crack, Investors Are Looking Elsew

NOTE: obb.portfolio_intel.sentiment router has a pre-existing
NameError on load — the generated static package references
`SentimentRollupResult` which isn't defined. Filed as a follow-up bug.

When that's fixed, this cell will call obb.portfolio_intel.sentiment
to get a per-name sentiment score aligned with the smart_money rollup.


## 7. From signal to trade rationale

At this point in a typical Sunday I have:

- **1-3 names with a hostile event** in the next 5-10 days
- **1-3 names with a meaningfully positive or negative smart-money
  score**
- **Maybe one divergence** where sentiment and smart-money point
  opposite ways

That's the raw material for NB05's three candidate trades. I don't
write NB05's trades here — I just save the signal work so NB05 can
pick them up.

> **📖 Event-driven** — a strategy family that trades around scheduled or announced events (earnings, M&A, spin-offs, index rebalances) rather than on directional macro views. This notebook does not run an event-driven strategy; it uses the same *inputs* (calendar + filings) as a decision overlay on a positional book. [Investopedia →](https://www.investopedia.com/terms/e/eventdriven.asp)

*The code cell below pickles the events + smart-money artifacts to
`.notebook_state/`.*

In [ ]:
# [Phase B / NB04 §6] Save events + smart_money artifacts for NB05
# Pickle safety same as NB02/NB03: trusted local only, gitignored,
# never shipped. Uses `# noqa: S403` per PR #1391 discipline.
import pickle  # noqa: S403  # trusted local artifact; safety documented
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)

events_artifact = {
    "basket": basket,
    "timeline": list(res.timeline) if hasattr(res, "timeline") else [],
    "by_symbol": dict(res.by_symbol) if hasattr(res, "by_symbol") else {},
    "warnings": list(res.warnings) if hasattr(res, "warnings") else [],
}

# Reuse the §3 rollup (live merged with the sanctioned example fixture
# when the sub-plan gap made the live rows empty).  This is what NB05
# reads as top_conviction, so it MUST match what we rendered above.
def _serialize(item):
    return {
        "symbol": item.symbol,
        "composite": float(item.composite),
        "by_source": dict(item.by_source),
        "signal_count": int(item.signal_count),
    }

smart_money_artifact = {
    "basket": basket,
    "by_symbol": {sym: _serialize(item) for sym, item in sm_by_symbol.items()},
    "top_conviction": [_serialize(item) for item in sm_top_conviction],
    "warnings": list(res.warnings) if res.warnings else [],
    "is_fixture": sm_is_fixture,
    "fixture_label": "example — signal shape as of 2026-06-30" if sm_is_fixture else None,
}

(state / "events.pkl").write_bytes(pickle.dumps(events_artifact))
(state / "smart_money.pkl").write_bytes(pickle.dumps(smart_money_artifact))

print(f"Wrote (repo-rel):  {str(state/'events.pkl'):<40}  {(state/'events.pkl').stat().st_size:,} bytes")
print(f"Wrote (repo-rel):  {str(state/'smart_money.pkl'):<40}  {(state/'smart_money.pkl').stat().st_size:,} bytes")
print(f"  top_conviction rows persisted: {len(smart_money_artifact['top_conviction'])} "
      f"({'fixture' if sm_is_fixture else 'live'})")
print()
print("NB05 will load these to build the 3 candidate trades from the signals.")


Wrote (repo-rel):  .notebook_state\events.pkl                851 bytes
Wrote (repo-rel):  .notebook_state\smart_money.pkl           1,381 bytes
  top_conviction rows persisted: 3 (fixture)

NB05 will load these to build the 3 candidate trades from the signals.


---

## What is NOT in this notebook

- **Dark-pool prints.** Would fit as a 4th smart-money stream; free-tier feeds don't exist yet.
- **Short-interest changes as a distinct signal.** Currently rolled into the 13F stream; deserves its own weight.
- **Options-flow (unusual activity).** Related to the offline options snapshot from NB01; not wired into `SmartMoneyScore` yet.

## Preview of NB05

Three names look wrong. But feelings about names aren't a trading plan. In NB05 we take the signal work above and turn it into three specific candidate trades — then diff the portfolio to see what those trades would actually *do* to my basket. If the diff shows concentration going up instead of down, we don't place the trade.


## 📚 Further reading

Every Investopedia link cited in this notebook:

- [Earnings announcement — Investopedia](https://www.investopedia.com/terms/e/earnings-announcement.asp)
- [Earnings surprise — Investopedia](https://www.investopedia.com/terms/e/earningssurprise.asp)
- [Ex-dividend date — Investopedia](https://www.investopedia.com/terms/e/ex-dividend.asp)
- [Reverse split — Investopedia](https://www.investopedia.com/terms/r/reversesplit.asp)
- [Form 13F — Investopedia](https://www.investopedia.com/terms/f/form-13f.asp)
- [Form 4 — Investopedia](https://www.investopedia.com/terms/f/form-4.asp)
- [Insider trading — Investopedia](https://www.investopedia.com/terms/i/insidertrading.asp)
- [STOCK Act — Investopedia](https://www.investopedia.com/terms/s/stock-act.asp)
- [Market sentiment — Investopedia](https://www.investopedia.com/terms/m/marketsentiment.asp)
- [Event-driven — Investopedia](https://www.investopedia.com/terms/e/eventdriven.asp)

**Notes on terms used bare (referenced elsewhere in the series):**

- *Stock split* — see NB01 §Corporate actions.
- *Dividend* — see NB01 §Corporate actions.

**Canonical references beyond Investopedia:**

- Ball, R. & Brown, P. — "An Empirical Evaluation of Accounting Income
  Numbers," *Journal of Accounting Research* 6(2), 1968. The original
  post-earnings-announcement drift paper; still the reference for why
  the calendar in §3 is a tradable overlay rather than pure noise.
- Cohen, L., Malloy, C. & Pomorski, L. — "Decoding Inside Information,"
  *Journal of Finance* 67(3), 2012. The empirical case for weighting
  routine vs. opportunistic insider trades differently — the intuition
  behind `SmartMoneyScore`'s per-role weighting in §4.


## 📚 Further reading

Every Investopedia link cited in this notebook:

- [Earnings announcement — Investopedia](https://www.investopedia.com/terms/e/earnings-announcement.asp)
- [Earnings surprise — Investopedia](https://www.investopedia.com/terms/e/earningssurprise.asp)
- [Ex-dividend date — Investopedia](https://www.investopedia.com/terms/e/ex-dividend.asp)
- [Reverse split — Investopedia](https://www.investopedia.com/terms/r/reversesplit.asp)
- [Form 13F — Investopedia](https://www.investopedia.com/terms/f/form-13f.asp)
- [Form 4 — Investopedia](https://www.investopedia.com/terms/f/form-4.asp)
- [Insider trading — Investopedia](https://www.investopedia.com/terms/i/insidertrading.asp)
- [STOCK Act — Investopedia](https://www.investopedia.com/terms/s/stock-act.asp)
- [Market sentiment — Investopedia](https://www.investopedia.com/terms/m/marketsentiment.asp)
- [Event-driven — Investopedia](https://www.investopedia.com/terms/e/eventdriven.asp)

**Notes on terms used bare (referenced elsewhere in the series):**

- *Stock split* — see NB01 §Corporate actions.
- *Dividend* — see NB01 §Corporate actions.

**Canonical references beyond Investopedia:**

- Ball, R. & Brown, P. — "An Empirical Evaluation of Accounting Income
  Numbers," *Journal of Accounting Research* 6(2), 1968. The original
  post-earnings-announcement drift paper; still the reference for why
  the calendar in §3 is a tradable overlay rather than pure noise.
- Cohen, L., Malloy, C. & Pomorski, L. — "Decoding Inside Information,"
  *Journal of Finance* 67(3), 2012. The empirical case for weighting
  routine vs. opportunistic insider trades differently — the intuition
  behind `SmartMoneyScore`'s per-role weighting in §4.
